In [1]:
# ─── Bibliotecas e classes ───

from pathlib import Path
from dotenv import find_dotenv

# ─── Raiz do projeto ───
PROJECT_ROOT = Path(find_dotenv(usecwd=True)).parent
EXTERNAL_DIR = PROJECT_ROOT / "data_lake" / "external"

In [ ]:
# ─── URL ───
MICRODADOS_INEP = {
    2023: "https://download.inep.gov.br/dados_abertos/microdados_avaliacao_da_alfabetizacao_2023.zip",
    2024: "https://download.inep.gov.br/dados_abertos/microdados_avaliacao_da_alfabetizacao_2024.zip",
    2025: "https://download.inep.gov.br/dados_abertos/microdados_AEEB_2025.zip",
}

print(f"📁 Destino dos downloads : {EXTERNAL_DIR}")
print(f"🔗 Anos disponíveis      : {list(MICRODADOS_INEP.keys())}")

In [7]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

def baixar_arquivo(url: str, destino: Path) -> Path:
    """Baixa um arquivo de uma URL para um caminho local, em streaming."""
    print(f"   ⬇️  Baixando: {destino.name} ...")
    with requests.get(url, headers=HEADERS, stream=True, timeout=30, verify=False) as resposta:
        resposta.raise_for_status()
        with open(destino, "wb") as f:
            for pedaco in resposta.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(pedaco)
    print(f"   ✅ {destino.name}")
    return destino

In [ ]:
for ano, url in MICRODADOS_INEP.items():
    destino = EXTERNAL_DIR / url.split("/")[-1]
    print(f"📅 Ano {ano}:")
    baixar_arquivo(url, destino)

📅 Ano 2023:
   ⬇️  Baixando: microdados_avaliacao_da_alfabetizacao_2023.zip ...
   ✅ microdados_avaliacao_da_alfabetizacao_2023.zip
📅 Ano 2024:
   ⬇️  Baixando: microdados_avaliacao_da_alfabetizacao_2024.zip ...
   ✅ microdados_avaliacao_da_alfabetizacao_2024.zip
📅 Ano 2025:
   ⬇️  Baixando: microdados_AEEB_2025.zip ...
   ✅ microdados_AEEB_2025.zip


In [5]:
# CSVs de dados que o pipeline usa (ignoramos TS_ITEM, dicionário, inputs, leia-me)
ARQUIVOS_ALVO = ["TS_ALUNO.csv", "TS_ESTADO.csv", "TS_MUNICIPIO.csv"]

# Pasta de destino dos CSVs extraídos, organizados por ano
EXTRAIDOS_DIR = EXTERNAL_DIR / "extraidos"

In [ ]:
import zipfile

def descompactar(ano: int, zip_path: Path) -> None:
    """Extrai os CSVs de dados de um ZIP do INEP para extraidos/{ano}/."""
    destino_ano = EXTRAIDOS_DIR / str(ano)
    destino_ano.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as z:
        for alvo in ARQUIVOS_ALVO:
            origem_interna = f"DADOS/{alvo}"          # caminho dentro do ZIP
            destino_arquivo = destino_ano / alvo       # caminho final, sem a subpasta DADOS/
            # Lê o arquivo de dentro do ZIP e escreve no destino, "achatando" a estrutura
            with z.open(origem_interna) as fonte, open(destino_arquivo, "wb") as saida:
                saida.write(fonte.read())
            print(f"   ✓ {ano}/{alvo}")

# Descompacta os três anos
for ano, url in MICRODADOS_INEP.items():
    zip_path = EXTERNAL_DIR / url.split("/")[-1]
    print(f"📦 Ano {ano}:")
    descompactar(ano, zip_path)

print(f"\n{'='*50}\n✅ CSVs extraídos em: {EXTRAIDOS_DIR}")

📦 Ano 2023:
   ✓ 2023/TS_ALUNO.csv
   ✓ 2023/TS_ESTADO.csv
   ✓ 2023/TS_MUNICIPIO.csv
📦 Ano 2024:
   ✓ 2024/TS_ALUNO.csv
   ✓ 2024/TS_ESTADO.csv
   ✓ 2024/TS_MUNICIPIO.csv
📦 Ano 2025:
   ✓ 2025/TS_ALUNO.csv
   ✓ 2025/TS_ESTADO.csv
   ✓ 2025/TS_MUNICIPIO.csv

✅ CSVs extraídos em: d:\diego\01_projects\postech-challenge-2\data_lake\external\extraidos


In [8]:
# Mapa das planilhas de metas (XLSX) — caminho e nomes diferentes dos ZIPs de microdados.
# Note o path distinto (/avaliacao_da_alfabetizacao/resultados/) e o sufixo de versão (_v1, _v2).
METAS_INEP = {
    "municipios_2023": "https://download.inep.gov.br/avaliacao_da_alfabetizacao/resultados_e_metas_municipios.xlsx",
    "ufs_2023":        "https://download.inep.gov.br/avaliacao_da_alfabetizacao/resultados_e_metas_ufs.xlsx",
    "municipios_2024": "https://download.inep.gov.br/alfabetiza_brasil/resultados_e_metas_municipios_2024.xlsx",
    "ufs_2024":        "https://download.inep.gov.br/alfabetiza_brasil/resultados_e_metas_ufs_2024_2.xlsx",
    "municipios_2025": "https://download.inep.gov.br/avaliacao_da_alfabetizacao/resultados/resultados_e_metas_municipios_2025_v2.xlsx",
    "ufs_2025":        "https://download.inep.gov.br/avaliacao_da_alfabetizacao/resultados/resultados_e_metas_ufs_2025_v1.xlsx",
}



for nome, url in METAS_INEP.items():
    # Extrai o ano da chave (ex.: municipios_2023 -> 2023)
    ano = nome.split("_")[-1]

    # Pasta do ano
    destino_dir = EXTRAIDOS_DIR / ano
    destino_dir.mkdir(parents=True, exist_ok=True)

    # Nome do arquivo
    destino = destino_dir / Path(url).name

    print(f"📅 {nome}:")
    baixar_arquivo(url, destino)


📅 municipios_2023:
   ⬇️  Baixando: resultados_e_metas_municipios.xlsx ...
   ✅ resultados_e_metas_municipios.xlsx
📅 ufs_2023:
   ⬇️  Baixando: resultados_e_metas_ufs.xlsx ...
   ✅ resultados_e_metas_ufs.xlsx
📅 municipios_2024:
   ⬇️  Baixando: resultados_e_metas_municipios_2024.xlsx ...
   ✅ resultados_e_metas_municipios_2024.xlsx
📅 ufs_2024:
   ⬇️  Baixando: resultados_e_metas_ufs_2024_2.xlsx ...
   ✅ resultados_e_metas_ufs_2024_2.xlsx
📅 municipios_2025:
   ⬇️  Baixando: resultados_e_metas_municipios_2025_v2.xlsx ...
   ✅ resultados_e_metas_municipios_2025_v2.xlsx
📅 ufs_2025:
   ⬇️  Baixando: resultados_e_metas_ufs_2025_v1.xlsx ...
   ✅ resultados_e_metas_ufs_2025_v1.xlsx
